# 03 — Zero-Shot Informative Classification (Qwen2.5-VL-7B-Instruct)

Run zero-shot classification on the **informative** task (2 classes: `informative`, `not_informative`) across all 3 modalities and both train/test splits.

| # | Modality | Split | Output |
|---|----------|-------|--------|
| 1 | text_only | test | `results/zeroshot/qwen2.5-vl-7b/informative/text_only/test/` |
| 2 | text_only | train | `results/zeroshot/qwen2.5-vl-7b/informative/text_only/train/` |
| 3 | image_only | test | `results/zeroshot/qwen2.5-vl-7b/informative/image_only/test/` |
| 4 | image_only | train | `results/zeroshot/qwen2.5-vl-7b/informative/image_only/train/` |
| 5 | text_image | test | `results/zeroshot/qwen2.5-vl-7b/informative/text_image/test/` |
| 6 | text_image | train | `results/zeroshot/qwen2.5-vl-7b/informative/text_image/train/` |

## Setup

In [1]:
import sys, os, gc
sys.path.insert(0, os.path.abspath(".."))

import torch
from scripts.zeroshot_qwen import load_model, run_zeroshot

# Configuration
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
TASK = "informative"
DATA_ROOT = os.path.abspath("../data")
OUTPUT_DIR = os.path.abspath("../results/zeroshot")

# Helper: clean up predictions and free GPU cache between experiments
def cleanup(*vars_to_del):
    """Delete prediction variables and free GPU memory."""
    for v in vars_to_del:
        del v
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Load model once — reused across all 6 experiments
model, processor = load_model(MODEL_ID)

Loading model: Qwen/Qwen2.5-VL-7B-Instruct


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Model loaded on cuda:0


## 1. Text-Only — Test Set

In [ ]:
preds, _ = run_zeroshot(
    task=TASK, modality="text_only", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 2. Text-Only — Train Set

In [ ]:
preds, _ = run_zeroshot(
    task=TASK, modality="text_only", split="train",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 3. Image-Only — Test Set

In [ ]:
preds, _ = run_zeroshot(
    task=TASK, modality="image_only", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 4. Image-Only — Train Set

In [ ]:
preds, _ = run_zeroshot(
    task=TASK, modality="image_only", split="train",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 5. Text+Image — Test Set

In [ ]:
preds, _ = run_zeroshot(
    task=TASK, modality="text_image", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## 6. Text+Image — Train Set

In [ ]:
preds, _ = run_zeroshot(
    task=TASK, modality="text_image", split="train",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)
cleanup(preds)

## Summary — All 6 Experiments

In [10]:
import json, os
import pandas as pd
from pathlib import Path

MODEL_SLUG = "qwen2.5-vl-7b"
TASK = "informative"
RESULTS_DIR = Path(os.path.abspath("../results/zeroshot")) / MODEL_SLUG / TASK

modalities = ["text_only", "image_only", "text_image"]
splits = ["test", "train"]
rows = []

for modality in modalities:
    for split in splits:
        metrics_path = RESULTS_DIR / modality / split / "metrics.json"
        if metrics_path.exists():
            m = json.loads(metrics_path.read_text())
            rows.append({
                "Modality": m["modality"],
                "Split": m["split"],
                "Samples": m["num_samples"],
                "Accuracy": f"{m['accuracy']:.4f}",
                "Weighted F1": f"{m['weighted_f1']:.4f}",
                "Macro F1": f"{m['macro_f1']:.4f}",
                "Unparseable": m["num_unparseable"],
            })
        else:
            rows.append({
                "Modality": modality, "Split": split,
                "Samples": "-", "Accuracy": "-",
                "Weighted F1": "-", "Macro F1": "-", "Unparseable": "-",
            })

print(f"Task: {TASK}")
print(f"Model: {MODEL_SLUG}\n")
display(pd.DataFrame(rows))

Task: informative
Model: qwen2.5-vl-7b



,Modality,Split,Samples,Accuracy,Weighted F1,Macro F1,Unparseable
0,text_only,test,1534,0.7027,0.7112,0.6975,0
1,text_only,train,8293,0.6859,0.6934,0.6827,0
2,image_only,test,1534,0.8729,0.8717,0.8533,0
3,image_only,train,9601,0.8727,0.8714,0.8553,0
4,text_image,test,1534,0.8403,0.8321,0.8025,0
5,text_image,train,9601,0.8422,0.8344,0.8085,0


## Cleanup — Free GPU Memory

In [ ]:
del model, processor

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.1f} GB")
print("\nNote: If memory is still high, restart the kernel to fully release GPU memory.")